In [1]:
# ============================
# Cell 1 — Imports
# ============================

import pandas as pd
from pathlib import Path

In [2]:
# ============================
# Cell 2 — USER SETTINGS (EDIT THESE)
# ============================

from pathlib import Path

# Folder containing your inputs and where outputs will be written
BASE = Path("/blue/juannanzhou/arindam.sarkar/BB-QTL_HT/Validation/Locator")

# Knockout library Excel files
HIP_XLSX = BASE / "Input/YSC1055_HIP_all_genes.tsv"   # HIP = heterozygous diploid (YSC1055)
HOP_XLSX = BASE / "Input/YSC1056_HOP_all_genes.tsv" # HOP = homozygous diploid (YSC1056)

# Output TSV files
OUT_HIP_TSV = BASE / "Output/HIP_all_genes_plate_locations.tsv"
OUT_HOP_TSV = BASE / "Output/HOP_all_genes_plate_locations.tsv"

print("BASE:", BASE)
print("HIP_XLSX:", HIP_XLSX)
print("HOP_XLSX:", HOP_XLSX)
print("OUT_HIP_TSV:", OUT_HIP_TSV)
print("OUT_HOP_TSV:", OUT_HOP_TSV)

BASE: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/Validation
HIP_XLSX: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/Validation/Input/YSC1055_HIP_all_genes.tsv
HOP_XLSX: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/Validation/Input/YSC1056_HOP_all_genes.tsv
OUT_HIP_TSV: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/Validation/Output/HIP_all_genes_plate_locations.tsv
OUT_HOP_TSV: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/Validation/Output/HOP_all_genes_plate_locations.tsv


In [5]:
# ============================
# Cell 3 — Load library TSV files
# ============================

def load_library_df(file_path: Path) -> pd.DataFrame:
    """
    Load a library TSV with columns:
        standard_name, orf, plate, row, col
    """

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    df = pd.read_csv(file_path, sep="\t")

    # Normalize column names
    df.columns = [str(c).strip().lower() for c in df.columns]

    required = {"standard_name", "orf", "plate", "row", "col"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"Missing columns in {file_path}: {missing}\n"
            f"Columns found: {list(df.columns)}"
        )

    out = df[["standard_name", "orf", "plate", "row", "col"]].copy()

    # Normalize strings
    out["standard_name"] = out["standard_name"].astype(str).str.strip().str.upper()
    out["orf"] = out["orf"].astype(str).str.strip().str.upper()

    # Drop empty rows
    out = out.dropna(subset=["standard_name", "orf", "plate", "row", "col"])

    # Remove duplicates
    out = out.drop_duplicates(subset=["orf", "plate", "row", "col"]).reset_index(drop=True)

    print(f"[load_library_df] {file_path.name} | rows={len(out):,}")

    return out

In [6]:
# ============================
# Cell 4 — Export full plate maps
# ============================

hip_df = load_library_df(HIP_XLSX)
hop_df = load_library_df(HOP_XLSX)

# Add library labels
hip_df["library"] = "YSC1055_HIP"
hop_df["library"] = "YSC1056_HOP"

# Sort
hip_out = hip_df[["standard_name", "orf", "library", "plate", "row", "col"]].sort_values(
    ["plate", "row", "col", "orf"], kind="mergesort"
)

hop_out = hop_df[["standard_name", "orf", "library", "plate", "row", "col"]].sort_values(
    ["plate", "row", "col", "orf"], kind="mergesort"
)

# Write TSV
hip_out.to_csv(OUT_HIP_TSV, sep="\t", index=False)
hop_out.to_csv(OUT_HOP_TSV, sep="\t", index=False)

print(f"\nWrote HIP TSV: {OUT_HIP_TSV} (rows={len(hip_out):,})")
print(f"Wrote HOP TSV: {OUT_HOP_TSV} (rows={len(hop_out):,})")

display(hip_out.head())
display(hop_out.head())

[load_library_df] YSC1055_HIP_all_genes.tsv | rows=6,512
[load_library_df] YSC1056_HOP_all_genes.tsv | rows=5,415

Wrote HIP TSV: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/Validation/Output/HIP_all_genes_plate_locations.tsv (rows=6,512)
Wrote HOP TSV: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/Validation/Output/HOP_all_genes_plate_locations.tsv (rows=5,415)


,standard_name,orf,library,plate,row,col
0,PAU8,YAL068C,YSC1055_HIP,201,A,2
1,SEO1,YAL067C,YSC1055_HIP,201,A,3
2,YAL066W,YAL066W,YSC1055_HIP,201,A,4
3,YAL065C,YAL065C,YSC1055_HIP,201,A,5
4,GDH3,YAL062W,YSC1055_HIP,201,A,6


,standard_name,orf,library,plate,row,col
0,ARN2,YHL047C,YSC1056_HOP,1,A,2
1,PAU13,YHL046C,YSC1056_HOP,1,A,3
2,PXP3,YHL045W,YSC1056_HOP,1,A,4
3,DFP4,YHL044W,YSC1056_HOP,1,A,5
4,ECM34,YHL043W,YSC1056_HOP,1,A,6


In [7]:
# ============================
# Final Cell — Merge HIP and HOP outputs
# ============================

import pandas as pd

# Read the exported files
hip_df = pd.read_csv(OUT_HIP_TSV, sep="\t")
hop_df = pd.read_csv(OUT_HOP_TSV, sep="\t")

# Add Library_type column
hip_df["Library_type"] = "HIP"
hop_df["Library_type"] = "HOP"

# Merge (concatenate)
merged_df = pd.concat([hip_df, hop_df], ignore_index=True)

# Optional: reorder columns if 'library' column exists
cols = merged_df.columns.tolist()

# Place Library_type after 'library' if present, else at the end
if "library" in cols:
    cols.remove("Library_type")
    insert_pos = cols.index("library") + 1
    cols = cols[:insert_pos] + ["Library_type"] + cols[insert_pos:]
    merged_df = merged_df[cols]

print("Merged rows:", len(merged_df))
print("Library_type counts:")
print(merged_df["Library_type"].value_counts())

display(merged_df.head())

Merged rows: 11927
Library_type counts:
Library_type
HIP    6512
HOP    5415
Name: count, dtype: int64


,standard_name,orf,library,Library_type,plate,row,col
0,PAU8,YAL068C,YSC1055_HIP,HIP,201,A,2
1,SEO1,YAL067C,YSC1055_HIP,HIP,201,A,3
2,YAL066W,YAL066W,YSC1055_HIP,HIP,201,A,4
3,YAL065C,YAL065C,YSC1055_HIP,HIP,201,A,5
4,GDH3,YAL062W,YSC1055_HIP,HIP,201,A,6


In [8]:
OUT_COMBINED = BASE / "Output/HIP_HOP_all_genes_plate_locations.tsv"
merged_df.to_csv(OUT_COMBINED, sep="\t", index=False)
print("Saved combined file:", OUT_COMBINED)

Saved combined file: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/Validation/Output/HIP_HOP_all_genes_plate_locations.tsv
